# PINN — General Postprocessing
## Compatible with NLS N=1, NLS N=2, Hirota N=1, Hirota N=2

---

**Structure:**
- **Cell 1** — Imports and global style
- **Cell 2** — Central configuration -> EDIT
- **Cell 3** — Internal helper functions
- **Cell 4** — General functions (field, loss, animation, striae)
- **Cell 5** — Summary and Spearman correlations
- **Cell 6** — Comparison functions across experiments
- **Cell 7** — Modular execution

**Individual functions (one experiment):**

| Function | Description |
|---------|-------------|
| `plot_field_error` | Field u/v/\|$\psi$\| (1x3: exact/PINN/error) |
| `plot_loss_total` | Total loss Adam+LBFGS |
| `plot_loss_components` | Loss by components IC/BC/Res |
| `animate_solution_vs_error` | Time animation solution vs. error |
| `plot_striae_isolines` | Error isolines (striae pattern) |
| `compute_spearman` | Spearman correlations (rho1–rho4b) |
| `plot_norm_conservation` | Conserved quantity N(t)=$\int\|\psi\|^2 dx$ |

**Comparative functions (one or more experiments):**

| Function | Description |
|---------|-------------|
| `summary_table` | CSV table with errors, times, iterations |
| `summary_spearman_table` | CSV summary table of correlations |
| `plot_loss_comparison` | Loss comparison across experiments |
| `plot_l2_instantaneous` | Comparison of the instantaneous L2 error |
| `plot_residual_heatmap` | Comparative heatmap of the PDE residual |
| `plot_striae_comparison` | Comparison of striae patterns |
| `plot_time_slices_comparison` | Comparative waterfall of time slices |
| `plot_sampling_panel` | Comparative panel of sampling strategies |

> **Note:** the functions accept a dictionary with one or more experiments
> alike — to plot a single experiment, pass a dictionary with
> a single entry.


In [ ]:
# ==============================================================================
# CONFIGURATION — must match the configuration used in the training notebook
# ==============================================================================
# The batch folder (BATCH_ID) and the experiment labels are rebuilt from these
# values, so copying the configuration cell of the training notebook is enough.
# `equation` is also consumed by plot_norm_conservation and compute_spearman.
#
# DEFAULT: Hirota N=1 fundamental soliton (Zhou & Yan, 2021), which runs with
# the reference data shipped in data/. The other cases need .mat files that
# are available on request (see data/README.md).
#
#   equation         case            .mat file                      x-domain   t-domain  in repo
#   'nls'            'fundamental'   NLS_N1.mat                     [-5, 5]    [0, pi/2]   yes
#   'nls'            'bound_state'   NLS_N2.mat                     [-5, 5]    [0, pi/2]   yes
#   'nls'            'collision'     NLS_2soliton_interaction.mat   [0, 20]    [0, 5]      request
#   'yan'            'fundamental'   Bright_soliton.mat             [-10, 10]  [0, 5]      yes
#   'hirota'         'bound_state'   hirota_N_2.mat                 [-5, 5]    [0, 2pi]    request
#   'hirota'         'collision'     N_2_hirota_inter.mat           [-10, 10]  [0, 4]      request
#
# equation = 'yan'    -> Hirota equation, Zhou & Yan (2021) parametrization (alpha=1, beta=0.01)
# equation = 'hirota' -> Hirota equation, Dr. L. Serkin's parametrization (D2, D3, R2, R3) = (1, 0.01, 1, 0.01)
# equation = 'nls'    -> standard NLS equation
# Domains follow Table 3.2 of the thesis.

equation     = 'yan'                     # 'yan' | 'hirota' | 'nls'
case         = 'fundamental'             # 'fundamental' | 'bound_state' (breather) | 'collision'
architecture = [2, 80, 80, 80, 80, 2]    # [input, hidden..., output]  (4 hidden layers x 80 neurons)
sampling_strategies = ['LHS', 'Sobol', 'Grid', 'Random']   # one experiment per strategy
bc_mode      = 1                         # boundary-condition mode:
                                         #   0, 1 : No BC
                                         #      2 : Val Only (periodic in psi)
                                         #      3 : Der1 Only (periodic in d_x psi)
                                         #      4 : Val + Der1
                                         #      5 : Val + Der1 + Der2
seed         = 1234                      # random seed (does not apply to Grid)
n_adam       = 5000                      # Adam epochs
n_lbfgs      = 10000                     # maximum L-BFGS iterations
lr           = 0.001                     # Adam learning rate

import numpy as np

CASES = {
    # (equation, case): (filename, lb=[x_min, t_min], ub=[x_max, t_max], short tag)
    ('nls',    'fundamental'): ('NLS_N1.mat',                   [-5.0, 0.0],  [5.0, np.pi/2], 'NLS_N1'),
    ('nls',    'bound_state'): ('NLS_N2.mat',                   [-5.0, 0.0],  [5.0, np.pi/2], 'NLS_N2Bound'),
    ('nls',    'collision'):   ('NLS_2soliton_interaction.mat', [0.0, 0.0],   [20.0, 5.0],    'NLS_N2Col'),
    ('yan',    'fundamental'): ('Bright_soliton.mat',           [-10.0, 0.0], [10.0, 5.0],    'Yan_N1'),
    ('hirota', 'bound_state'): ('hirota_N_2.mat',               [-5.0, 0.0],  [5.0, 2*np.pi], 'Hir_N2Bound'),
    ('hirota', 'collision'):   ('N_2_hirota_inter.mat',         [-10.0, 0.0], [10.0, 4.0],    'Hir_N2Col'),
}
if (equation, case) not in CASES:
    raise ValueError(f'No reference data for equation={equation!r}, case={case!r}. '
                     f'Valid combinations: {sorted(CASES)}')

data_file, lb_case, ub_case, case_tag = CASES[(equation, case)]
lb_case = np.array(lb_case)
ub_case = np.array(ub_case)

arch_tag = f'{len(architecture) - 2}x{architecture[1]}'
bc_tag   = {0: 'NoBC', 1: 'NoBC', 2: 'ValOnly', 3: 'Der1Only',
            4: 'ValDer1', 5: 'ValDer1Der2'}[bc_mode]
BATCH_ID = f'{case_tag}_{arch_tag}_{bc_tag}_Sampling'   # outputs/<BATCH_ID>/ and results/<BATCH_ID>/


def make_exp_id(strategy):
    """Experiment identifier; it is also the stem of the output file names."""
    return f'{case_tag}_{arch_tag}_{strategy}_{n_adam / 1000:g}kAdam_{n_lbfgs / 1000:g}kFGS_{bc_tag}'


print(f'Case     : equation={equation!r}, case={case!r} -> {data_file}')
print(f'Domain   : x in [{lb_case[0]:g}, {ub_case[0]:g}], t in [{lb_case[1]:g}, {ub_case[1]:.4g}]')
print(f'Batch ID : {BATCH_ID}')


In [ ]:
# ============================================================
# CELL 1: IMPORTS AND GLOBAL STYLE
# ============================================================

!pip install cmocean -q # Colormap

import os
import sys
import json
import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import pandas as pd
import cmocean
import seaborn as sns
from matplotlib.colors import LogNorm, TwoSlopeNorm

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust to the path where the repo was cloned/copied inside Drive:
    REPO_PATH = '/content/drive/MyDrive/PINN_NLS_Hirota'
else:
    # Local execution (VS Code, Terminal, Jupyter, etc.). Assumes the notebook is run from notebooks/
    REPO_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))

# --- External computation modules ------------------------------------
sys.path.append(os.path.join(REPO_PATH, 'src'))
from conservation import (compute_norm, compute_J_hirota,
                           compute_J_nls, check_boundary_flux)
from spearman      import (spearman_safe, compute_gradient_magnitude,
                            compute_dist_nn)

# --- Paths for reading data and writing figures/tables --------------
OUTPUTS_PATH = os.path.join(REPO_PATH, 'outputs', BATCH_ID)   # written by the training notebook
SAVE_PATH    = os.path.join(REPO_PATH, 'results', BATCH_ID)   # figures and tables of this batch
os.makedirs(SAVE_PATH, exist_ok=True)
if not os.path.exists(OUTPUTS_PATH):
    raise FileNotFoundError(
        f"{OUTPUTS_PATH} not found. "
        "Run the training notebook (PINN_Torch_NLS_Hirota.ipynb) first with the "
        "same configuration."
    )

# --- Global style (serif + Computer Modern) ----------------
plt.rcParams.update({
    'font.family':       'serif',
    'mathtext.fontset':  'cm',
    'axes.unicode_minus': False,
    'font.size':          14,
    'axes.labelsize':     14,
    'axes.titlesize':     15,
    'xtick.labelsize':    12,
    'ytick.labelsize':    12,
    'legend.fontsize':    12,
    'figure.titlesize':   16,
    'lines.linewidth':    1.6,
    'axes.grid':          True,
    'grid.alpha':         0.25,
    'grid.linewidth':     0.6,
    'figure.dpi':         120,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
})

PALETTE = [
    '#E69F00',
    '#0072B2',
    '#009E73',
    '#D55E00',
    '#CC79A7',
    '#F0E442',
    '#56B4E9',
    '#000000',
]

In [ ]:
# =================================================================================
# CELL 2: CENTRAL CONFIGURATION
# =================================================================================
# FIELDS OF experiment_params:
#   'label' : str — must exactly match the exp_id of the .npz
#   'desc'  : str — human-readable description for legends
#
#   equation : 'hirota', 'yan' or 'nls' — used in plot_norm_conservation and spearman
#              (defined in the configuration cell at the top of the notebook)
#   The labels are rebuilt with make_exp_id(); edit the list by hand to load
#   experiments with other names.
# =================================================================================

STRATEGY_DESC = {
    'LHS':    'Latin Hypercube (LHS)',
    'Sobol':  'Sobol sequence',
    'Grid':   'Uniform grid (Grid)',
    'Random': 'Uniform random (Random)',
}
experiment_params = [
    {'label': make_exp_id(s), 'desc': STRATEGY_DESC[s]}
    for s in sampling_strategies
]

# --- Loading results --------------------------------------------
results       = []
loaded_params = []

for exp in experiment_params:
    path = os.path.join(OUTPUTS_PATH, f"PINN_results_{exp['label']}.npz")
    if os.path.exists(path):
        with np.load(path, allow_pickle=True) as data:
            results.append({k: np.array(data[k]) for k in data.files})
        loaded_params.append(exp)
        print(f"Loaded: {exp['label']}")
    else:
        print(f"not found: {path}")

n_exp  = len(results)
labels = [p['label'] for p in loaded_params]
descs  = [p['desc']  for p in loaded_params]
colors = PALETTE[:n_exp]
print(f'Experiments loaded: {n_exp}')
if n_exp == 0:
    raise FileNotFoundError(f'No results found in {OUTPUTS_PATH}. Check that the '
                            'configuration cell matches the training run.')

In [ ]:
# ============================================================
# CELL 3: INTERNAL HELPER FUNCTIONS
# ============================================================

def _optimizer_labels(ax, adam_iters):
    ax.axvline(adam_iters, color='#555555', ls=':', lw=1.2)
    x_min, x_max = ax.get_xlim()
    dx    = 0.04 * (x_max - x_min)
    trans = ax.get_xaxis_transform()
    ax.text(adam_iters - dx, 0.05, 'Adam',
            transform=trans, fontsize=11, color='#444444',
            ha='right', va='center', fontstyle='italic')
    ax.text(adam_iters + dx, 0.05, 'L-BFGS',
            transform=trans, fontsize=11, color='#444444',
            ha='left',  va='center', fontstyle='italic')


def _save_fig(fig, save_path, filename):
    os.makedirs(save_path, exist_ok=True)
    for ext in ['pdf', 'png']:
        fig.savefig(os.path.join(save_path, f'{filename}.{ext}'))
    print(f'  saved: {filename}')


print('Internal helper functions defined.')

In [ ]:
# ============================================================
# CELL 4:    General functions (single experiment or comparative).
#             plot_field_error, plot_loss_total, plot_loss_components,
#             plot_loss_comparison, animate_solution_vs_error,
#             plot_striae_isolines
# ============================================================

def plot_field_error(res, lbl, desc, fig_path,
                     field='h', plot_collocation=False):
    """
    Figure 1x3: exact / PINN / absolute error (log scale).
    field: 'u', 'v' or 'h' (|psi|).
    Output: C_field_{field}_{lbl}.pdf / .png
    """
    x_vals = res['x'].flatten()
    t_vals = res['t'].flatten()
    extent = [x_vals.min(), x_vals.max(), t_vals.min(), t_vals.max()]
    field_map = {
        'u': (
            res['U_star'], res['U_pred'], float(res['error_u']),
            r'$u(x,t)$',           # name for suptitle
            r'$u(x,t)$',           # exact
            r'$|\hat{u}(x,t;\boldsymbol{\theta^{*}})|$',  # PINN
            r'$|\hat{u}(x,t;\boldsymbol{\theta^{*}})-u(x,t)|$', # error
            cmocean.cm.balance, True
        ),
        'v': (
            res['V_star'], res['V_pred'], float(res['error_v']),
            r'$v(x,t)$',           # name for suptitle
            r'$v(x,t)$',           # exact
            r'$\hat{v}(x,t;\boldsymbol{\theta^{*}})$',  # PINN
            r'$|\hat{v}(x,t;\boldsymbol{\theta^{*}})-v(x,t)|$', # error
            cmocean.cm.balance, True
        ),
        'h': (
            res['H_star'], res['H_pred'], float(res['error_h']),
            r'$|\psi(x,t)|$', #name for suptitle
            r'$|\psi(x,t)|$', #exact
            r'$|\hat{\psi}(x,t;\boldsymbol{\theta^{*}})|$', #PINN
            r'$|\,|\hat{\psi}(x,t;\boldsymbol{\theta^{*}})|-|\psi(x,t)|\,|$', #error
            cmocean.cm.haline_r, False
        )
    }

    (exact, pred, err_val, name, exact_title,
    pred_title, err_title, cmap_sol, use_div) = field_map[field]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                            sharex=True, sharey=True,
                            constrained_layout=True)

    ims =[]
    data_list = [(exact, exact_title),
                 (pred, pred_title),
                 (np.abs(exact - pred), err_title)
                 ]

    if use_div:
        # fields centered at 0 (u, v)
        vmin = min(exact.min(), pred.min())
        vmax = max(exact.max(), pred.max())
        vmax = max(abs(vmin), abs(vmax))
        vmin = -vmax
        norm_sol = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    else:
        # positive fields (h) -> log scale
        all_data = np.concatenate([exact.ravel(), pred.ravel()])
        pos_data = all_data[all_data > 0]  # ignore zeros
        vmin = pos_data.min() if pos_data.size else 1e-12
        vmax = all_data.max()
        norm_sol = LogNorm(vmin=vmin, vmax=vmax)

    for i, (data, title) in enumerate(data_list):
        ax = axes[i]
        if i < 2:
            im = ax.imshow(data, origin='lower', aspect='auto',
                           extent=extent, cmap=cmap_sol, norm=norm_sol)
        else:
            pos = data[data > 0]
            ve  = pos.min() if pos.size else 1e-12
            im  = ax.imshow(data, origin='lower', aspect='auto',
                            extent=extent, cmap=cmocean.cm.thermal_r,
                            norm=LogNorm(vmin=ve, vmax=data.max()))
        ims.append(im)
        if plot_collocation and 'X_f_train' in res:
            Xf = res['X_f_train']
            ax.scatter(Xf[:,1], Xf[:,0], s=1, c='white',
                       alpha=0.8, linewidths=0)
        ax.set_title(title); ax.set_xlabel(r'$x$',fontsize=14)
        if i == 0: ax.set_ylabel(r'$t$', fontsize=14)
    fig.colorbar(ims[0], ax=axes[:2], fraction=0.025, pad=0.02)
    fig.colorbar(ims[2], ax=axes[2],  fraction=0.025, pad=0.02)
    _save_fig(fig, fig_path, f'A_field_{field}_{lbl}')
    plt.show(); plt.close(fig)

def plot_loss_total(res, lbl, desc, fig_path):
    """
    Total loss on a semilogarithmic scale.
    Adam/L-BFGS split taken from adam_iters in the .npz.
    Output: D_loss_{lbl}.pdf / .png
    """
    loss = res['loss_total']
    ai   = int(res['adam_iters'][0])
    n    = len(loss); x = np.arange(n)
    margin = max(int(0.05*n), 2000)
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    ax.semilogy(x[:ai], loss[:ai], color=PALETTE[4], lw=1.8, label='Adam')
    ax.semilogy(x[ai:], loss[ai:], color=PALETTE[4], lw=1.8, ls='--', label='L-BFGS')
    ax.set_xlim(-margin, n+margin)
    _optimizer_labels(ax, ai)
    ax.set_xlabel('Iterations'); ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'Total loss history'); ax.legend()
    _save_fig(fig, fig_path, f'B_loss_{lbl}')
    plt.show(); plt.close(fig)

def plot_loss_components(res, lbl, desc, fig_path):
    comps = [
        (res['loss_res'], PALETTE[0], 'PDE residual'),
        (res['loss_ic'],  PALETTE[3], 'Initial condition (IC)'),
        (res['loss_bc'],  PALETTE[2], 'Boundary condition (BC)'),
    ]
    ai = int(res['adam_iters'][0]); n = len(res['loss_ic'])
    x = np.arange(n); margin = max(int(0.05*n), 200)
    fig, axes = plt.subplots(3, 1, figsize=(12, 10),
                             sharex=True, constrained_layout=True)
    fig.suptitle(f'Loss by components - {desc}')
    for ax, (vals, col, name) in zip(axes, comps):
        ax.semilogy(x[:ai], vals[:ai], color=col, lw=1.5, label=name)
        ax.semilogy(x[ai:], vals[ai:], color=col, lw=1.5, ls='--')
        ax.set_xlim(-margin, n+margin); _optimizer_labels(ax, ai)
        ax.set_ylabel('Loss (MSE)'); ax.set_title(name)
        ax.legend(loc='upper right')
    axes[-1].set_xlabel('Iterations (cumulative)')
    _save_fig(fig, fig_path, f'C_loss_components_{lbl}'); plt.show(); plt.close(fig)

def plot_loss_comparison(results, loaded_params,fig_path,
                         filename='D_loss_comparison'):
    """
    Comparative total loss for all experiments.
    Useful to compare convergence across architectures.
    Output: {filename}.pdf / .png
    """
    n = len(results)
    colors = sns.color_palette('colorblind', n)

    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    for res, exp, col in zip(results, loaded_params, colors):
        loss = res['loss_total']
        ai   = int(res['adam_iters'][0])
        n    = len(loss)
        x    = np.arange(n)
        lbl  = exp.get('desc', exp['label'])
        ax.semilogy(x[:ai], loss[:ai], color=col, lw=1.6, label=lbl)
        ax.semilogy(x[ai:], loss[ai:], color=col, lw=1.6, ls='--')

    ai_ref = int(results[0]['adam_iters'][0])
    n_ref  = len(results[0]['loss_total'])
    margin = max(int(0.05 * n_ref), 200)
    ax.set_xlim(-margin, n_ref + margin)
    _optimizer_labels(ax, ai_ref)
    ax.set_xlabel(r'Iterations')
    ax.set_ylabel(r'Total loss (MSE)')
    ax.set_title(r'Total loss comparison per experiment')
    ax.legend(title='Experiment', framealpha=0.95, fontsize=10)
    _save_fig(fig, fig_path, filename)
    plt.show(); plt.close(fig)

def animate_solution_vs_error(res, lbl, desc, fig_path,
                              fps=5, filename='E_solution_vs_error'):
    """
    MP4: profile of |psi(x,t)| exact vs PINN + absolute error evolving in t.
    Requires: !apt-get install -y ffmpeg
    Output: {filename}_{lbl}.mp4
    """
    import subprocess
    import matplotlib.animation as animation
    subprocess.run(['apt-get','install','-y','ffmpeg','-qq'], capture_output=True)
    x_vals = res['x'].flatten(); t_vals = res['t'].flatten()
    H_star = res['H_star'];      H_pred = res['H_pred']
    error  = np.abs(H_star - H_pred)
    pos    = error[error > 0]
    vmin_e = pos.min() if pos.size else 1e-12
    vmax_e = error.max()
    h_max  = max(H_star.max(), H_pred.max()) * 1.05
    vid    = os.path.join(fig_path, f'{filename}_{lbl}.mp4')
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=fps, metadata=dict(artist='PINN'), bitrate=2400)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    with writer.saving(fig, vid, dpi=200):
        for k, t_val in enumerate(t_vals):
            axes[0].cla(); axes[1].cla()
            axes[0].plot(x_vals, H_star[k,:], 'k-', lw=2.0, label='Exact')
            axes[0].plot(x_vals, H_pred[k,:], '--', color=PALETTE[1], lw=1.8, label='PINN')
            axes[0].set_xlim(x_vals.min(), x_vals.max()); axes[0].set_ylim(0, h_max)
            axes[0].set_xlabel('x'); axes[0].set_ylabel(r'$|\psi(x,t)|$')
            axes[0].set_title(f'Solution   t = {t_val:.3f}')
            axes[0].legend(loc='upper right'); axes[0].grid(True, alpha=0.25)
            axes[1].semilogy(x_vals, error[k,:], color=PALETTE[3], lw=1.8)
            axes[1].set_xlim(x_vals.min(), x_vals.max())
            axes[1].set_ylim(vmin_e, vmax_e)
            axes[1].set_xlabel('x'); axes[1].set_ylabel(r'$\|\hat{\psi}(x,t;\boldsymbol{\theta}^{*}) - \psi(x,t)\|$')
            axes[1].set_title(f'Absolute error   t = {t_val:.3f}')
            axes[1].grid(True, which='both', alpha=0.25)
            fig.suptitle(f'{desc}'); writer.grab_frame()
    plt.close(fig); print(f'Video saved: {vid}')

def plot_striae_isolines(res, lbl, desc, fig_path,
                         n_levels=8, filename_suffix=''):
    x_vals = res['x'].flatten(); t_vals = res['t'].flatten()
    H_star = res['H_star'];      H_pred = res['H_pred']
    error  = np.abs(H_star - H_pred)
    extent = [x_vals.min(), x_vals.max(), t_vals.min(), t_vals.max() ]
    pos_err  = error[error > 0]
    vmin_err = pos_err.min() if pos_err.size else 1e-12
    norm_err = LogNorm(vmin=vmin_err, vmax=error.max())
    h_min = H_star.min(); h_max = H_star.max()
    pos_h = H_star[H_star > 0]
    vmin_h = pos_h.min() if pos_h.size else 1e-12
    norm_h = LogNorm(vmin=vmin_h, vmax=h_max)
    levels = np.linspace(h_min+0.05*(h_max-h_min),
                         h_max-0.05*(h_max-h_min), n_levels) # 0.0005
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             sharex=True, sharey=True, constrained_layout=True)
    ax0 = axes[0]
    im0 = ax0.imshow(error, origin='lower', aspect='auto', extent=extent,
                     cmap=cmocean.cm.thermal_r, norm=norm_err)
    cs0 = ax0.contour(x_vals, t_vals, H_star, levels=levels,
                      colors='white', linewidths=0.8, alpha=0.85)
    ax0.clabel(cs0, fmt='%.2f', fontsize=9, colors='white')
    ax0.set_xlabel(r'$x$'); ax0.set_ylabel(r'$t$')
    ax0.set_title(
        r'$\left|\,|\hat{\psi}(x,t;\boldsymbol{\theta}^{*})|-|\psi(x,t)|\,\right|$'
        r' + isolines $|\psi(x,t)|$'
    )
    fig.colorbar(im0, ax=ax0, fraction=0.03, pad=0.02)
    ax1 = axes[1]
    im1 = ax1.imshow(H_star, origin='lower', aspect='auto', extent=extent,
                    norm=norm_h, cmap=cmocean.cm.haline_r)
    cs1 = ax1.contour(x_vals, t_vals, H_star, levels=levels,
                      colors='black', linewidths=0.8, alpha=0.85)
    ax1.clabel(cs1, fmt='%.2f', fontsize=9, colors='black')
    ax1.set_xlabel(r'$x$'); ax1.set_title(r'$|\psi(x,t)|$')
    fig.colorbar(im1, ax=ax1, fraction=0.03, pad=0.02)
    fname = f'F_striae_isolines_{lbl}{filename_suffix}'
    _save_fig(fig, fig_path, fname); plt.show(); plt.close(fig)


In [ ]:
# ============================================================
# CELL 5 — plot_l2_bars, summary_table,
#             summary_spearman_table, plot_norm_conservation,
#             plot_l2_instantaneous
# ============================================================

def summary_table(results, loaded_params, fig_path,
                  filename='G_summary_table'):
    """
    CSV: L2 for u/v/|q|, time and actual iterations.
    Output: {filename}.csv
    """
    rows = []
    for res, exp in zip(results, loaded_params):
        ai = int(res['adam_iters'][0]); ni = len(res['loss_total'])
        rows.append({
            'Experimento': exp['desc'],
            'L2 |q|':      f"{float(res['error_h']):.3e}",
            'L2 u':        f'{float(res['error_u']):.3e}',
            'L2 v':        f'{float(res['error_v']):.3e}',
            'Tiempo (min)':  f'{float(res['elapsed_time'])/60:.1f}',
            'Iter Adam':   ai, 'Iter LBFGS': ni-ai, 'Iter Total': ni,
            'Final Residual Loss': f'{float(res['loss_res'][-1]):.3e}',
        })
    df = pd.DataFrame(rows)
    os.makedirs(fig_path, exist_ok=True)
    df.to_csv(os.path.join(fig_path, f'{filename}.csv'), index=False)
    print(df.to_string(index=False)); return df

def compute_spearman(res, lbl, desc,
                     compute_gradient_magnitude, equation='nls'):
    """
    Computes Spearman correlations between the pointwise absolute error
    and four proxies. It does not generate any figure.

    Proxies
    -------
    rho_1  Amplitude |q_exact|         — expected negative (well-learned crests)
    rho_2  Gradient |d_x q_exact|      — expected positive (variation -> more error)
    rho_3  Distance to the nearest     — geometry of the T_f sampling;
           collocation point             central proxy in the sampling study
    rho_4  PDE residual F_mag          — if significantly > 0: the PINN self-assesses
                                          without a reference solution

    Note on large n
    ---------------
    With ~20 000 points, small values of rho are statistically
    significant even if they are physically negligible.
    Threshold of practical relevance: |rho| > 0.3.
    Always report magnitude + p-value.

    Parameters
    ----------
    res                        : dict — loaded from .npz
                                Must include X_f_train for rho_3.
    lbl                        : str  — experiment id
    desc                       : str  — human-readable description
    compute_gradient_magnitude : func — notebook helper
    equation                   : str  — 'nls' | 'yan' | 'hirota'
                                Controls the FU/FV swap correction for
                                rho_4a, rho_4b.

    Returns
    -------
    dict with keys: label, desc, rho1..rho4, rho4a, rho4b, their p-values,
    error_u, error_v, error_h_L2 and internal arrays (_...) for reuse.
    """
    x_vals = res['x'].flatten()
    t_vals = res['t'].flatten()
    U_star = res['U_star']; V_star = res['V_star']
    U_pred = res['U_pred']; V_pred = res['V_pred']
    FU_pred = res['FU_pred']; FV_pred = res['FV_pred']
    X_f_train = res['X_f_train']

    err_u = np.abs(U_star - U_pred); ef_u = err_u.ravel()
    err_v = np.abs(V_star - V_pred); ef_v = err_v.ravel()

    H_star = res['H_star']
    H_pred = res['H_pred']
    F_mag  = np.sqrt(FU_pred**2 + FV_pred**2)
    error  = np.abs(H_star - H_pred)
    grad   = compute_gradient_magnitude(H_star, x_vals)
    dist   = compute_dist_nn(X_f_train, x_vals, t_vals)
    ef     = error.ravel()

    rho1, p1 = spearman_safe(H_star.ravel(), ef)
    rho2, p2 = spearman_safe(grad.ravel(),   ef)
    rho3, p3 = spearman_safe(dist.ravel(),   ef)
    rho4, p4 = spearman_safe(F_mag.ravel(),  ef)

    if equation == 'hirota':
        rho4a, p4a = spearman_safe(-FV_pred.ravel(), ef_u)
        rho4b, p4b = spearman_safe( FU_pred.ravel(), ef_v)
    else:
        rho4a, p4a = spearman_safe(FU_pred.ravel(), ef_u)
        rho4b, p4b = spearman_safe(FV_pred.ravel(), ef_v)

    print(f'  [{lbl}]  '
          f'rho_1={rho1:+.4f}   (p={p1:.2e})  '
          f'rho_2={rho2:+.4f}   (p={p2:.2e})  '
          f'rho_3={rho3:+.4f}   (p={p3:.2e})  '
          f'rho_4={rho4:+.4f}   (p={p4:.2e})  '
          f'rho_4a={rho4a:+.4f}  (p={p4a:.2e})  '
          f'rho_4b={rho4b:+.4f}  (p={p4b:.2e})  ')

    return {
        'label'      : lbl,
        'desc'       : desc,
        'rho1'       : rho1, 'p1': p1,
        'rho2'       : rho2, 'p2': p2,
        'rho3'       : rho3, 'p3': p3,
        'rho4'       : rho4, 'p4': p4,
        'rho4a'      : rho4a, 'p4a': p4a,
        'rho4b'      : rho4b, 'p4b': p4b,
        'error_u'    : float(res['error_u']),
        'error_v'    : float(res['error_v']),
        'error_h_L2' : float(res['error_h']),
        # arrays for reuse in plot_spearman_scatter without recomputing
        '_H_star'    : H_star.ravel(),
        '_U_pred'    : U_pred.ravel(),
        '_V_pred'    : V_pred.ravel(),
        '_grad'      : grad.ravel(),
        '_dist'      : dist.ravel(),
        '_F_mag'     : F_mag.ravel(),
        '_error'     : ef,
        '_ef_u'      : ef_u,
        '_ef_v'      : ef_v,
    }

def summary_spearman_table(tabla_resultados, fig_path,
                           filename='H_spearman_summary'):
    """
    Consolidates compute_spearman dicts into CSV and JSON.

    The internal arrays (_H_star, _grad, _dist, etc.) are excluded
    from the JSON to keep the file readable. The p-values are
    saved in the JSON but not in the CSV.
    """
    tabla_clean = [
        {k: v for k, v in d.items() if not k.startswith('_')}
        for d in tabla_resultados
    ]

    df = pd.DataFrame(tabla_clean).set_index('desc')
    cols = {
        'rho1'       : 'rho1(amplitud)',
        'p1'         : 'p1(amplitud)',
        'rho2'       : 'rho2(gradiente)',
        'p2'         : 'p2(gradiente)',
        'rho3'       : 'rho3(dist_NN)',
        'p3'         : 'p3(dist_NN)',
        'rho4'       : 'rho4(residuo)',
        'p4'         : 'p4(residuo)',
        'rho4a'      : 'rho4a(residuo FU)',
        'p4a'        : 'p4a(residuo FU)',
        'rho4b'      : 'rho4b(residuo FV)',
        'p4b'        : 'p4b(residuo FV)',
        'error_h_L2' : 'Error L2 |psi|',
    }
    d = df[list(cols)].rename(columns=cols)

    print('\nSpearman correlation table:')
    print(d.to_string(float_format='{:+.4f}'.format))

    os.makedirs(fig_path, exist_ok=True)
    d.to_csv(os.path.join(fig_path, f'{filename}.csv'),
             float_format='%.4e')
    with open(os.path.join(fig_path, f'{filename}.json'), 'w') as f:
        json.dump(tabla_clean, f, indent=4)
    print(f'Saved to {fig_path}')

    return d

def plot_norm_conservation(results, loaded_params, fig_path,
                           equation='hirota',
                           show_boundary_flux=True,
                           filename='I_norm_conservation'):
    """
    Conservation diagnostic of N(t) = integral |psi|^2 dx.

    The third panel (boundary flux) is the central diagnostic
    for the case with no imposed BC: it allows verifying
    numerically that the domain is wide enough.

    TERMINOLOGICAL DISTINCTION:
      N(t)       — conserved quantity (spatial integral of |psi|^2)
      ΔN_rel(t)  — relative deviation PINN vs exact
      error_L2   — global pointwise scalar (in summary_table)
    """

    n = len(results)
    colors = sns.color_palette('colorblind', n)
    n_panels = 3 if show_boundary_flux else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6*n_panels, 5),
                             constrained_layout=True)
    ax_N = axes[0]; ax_dN = axes[1]

    print('\n── Conservation diagnostic of N(t) ──')
    first = True
    for i, (res, exp) in enumerate(zip(results, loaded_params)):
        x_vals = res['x'].flatten(); t_vals = res['t'].flatten()
        col    = colors[i % len(colors)]
        desc   = exp.get('desc', f'exp{i}')

        N_exact  = compute_norm(res['U_star'], res['V_star'], x_vals)
        N_pred   = compute_norm(res['U_pred'], res['V_pred'], x_vals)
        dN_rel   = np.abs(N_pred - N_exact) / (N_exact + 1e-12)
        cv_exact = N_exact.std() / (N_exact.mean() + 1e-12)

        if first:
            ax_N.plot(t_vals, N_exact, 'k--', lw=1.6, alpha=0.6,
                      label=r'$\mathcal{N}$')
            first = False
        ax_N.plot(t_vals, N_pred,  color=col, lw=1.6, label=desc)
        ax_dN.plot(t_vals, dN_rel, color=col, lw=1.6, label=desc)

        print(f'  [{exp["label"]}]  CV(N)={cv_exact:.2e}  '
              f'{"✓" if cv_exact < 1e-3 else "⚠ check domain"}  '
              f'min(ΔN_rel)={dN_rel.min():.2e}'
              f'max(ΔN_rel)={dN_rel.max():.2e}')

    if show_boundary_flux:
        ax_J = axes[2]

        # The exact solution is identical for all experiments,
        # so the boundary flux is computed only once.
        res0 = results[0]

        x_vals = res0['x'].flatten()
        t_vals = res0['t'].flatten()

        print('\n  Boundary flux (exact solution):')

        flux, _, _ = check_boundary_flux(res0['U_star'], res0['V_star'], x_vals,
                                         equation=equation)
        ax_J.plot(t_vals, flux, color='k', lw=1.8, label='Exact', alpha=0.6)
        ax_J.axhline(0, color='gray', lw=0.8, ls=':')
        ax_J.set_xlabel(r'$t$')
        ax_J.set_ylabel(r'$J(+L,t) - J(-L,t)$')
        ax_J.set_title('Residual boundary flux\n(exact solution)')
        ax_J.legend(fontsize=10)

    ax_N.set_xlabel(r'$t$')
    ax_N.set_ylabel(r'$\hat{\mathcal{N}}(t) = \int |\psi|^2\,dx$')
    ax_N.set_title(r'Conserved quantity $\mathcal{N}(t)$')
    ax_N.legend(fontsize=10, framealpha=0.95)
    ax_dN.set_xlabel(r'$t$')
    ax_dN.set_ylabel(
        r'$\Delta\mathcal{N}_\mathrm{rel}(t)$', fontsize=10)
    ax_dN.set_title(r'Relative deviation of $\mathcal{N}$')
    ax_dN.set_yscale('log'); ax_dN.legend(fontsize=10, framealpha=0.95)
    _save_fig(fig, fig_path, filename); plt.show(); plt.close(fig)

def plot_l2_instantaneous(results, loaded_params, fig_path,
                          filename='J_l2_instantaneous'):
    """
    L2(t) = ||H_pred(.,t) - H_star(.,t)||_2 / ||H_star(.,t)||_2
    Time series of the instantaneous L2 error (different from the scalar global L2).
    """
    n = len(results)
    colors = sns.color_palette('colorblind', n)

    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    t_vals = results[0]['t'].flatten()
    for res, exp, col in zip(results, loaded_params, colors):
        num  = np.linalg.norm(res['H_pred'] - res['H_star'], axis=1)
        den  = np.linalg.norm(res['H_star'], axis=1)
        l2_t = num / (den + 1e-12)
        ax.semilogy(t_vals, l2_t, color=col, lw=1.6,
                    label=exp.get('desc', f'exp{loaded_params.index(exp)}'))
    ax.set_xlabel(r'$t$')
    ax.set_ylabel(r'$\varepsilon_{L2}(t)[|\psi|]$')
    ax.set_title(r'Instantaneous relative $L_2$ error per experiment')
    ax.grid(True, which='both', alpha=0.3)
    experiment_lines = [plt.Line2D([0],[0], color=colors[i], lw=2) for i in range(n)]
    experiment_labels = [exp.get('desc', f'exp{i}') for i, exp in enumerate(loaded_params)]
    fig.legend(
    experiment_lines, experiment_labels,
    bbox_to_anchor=(1.02, 0.5),  # outside, to the right of the right panel
    loc='center left',
    borderaxespad=0.,
    title='Strategy' #Modify according to the experiment
    )
    _save_fig(fig, fig_path, filename); plt.show(); plt.close(fig)

In [ ]:
# ============================================================
# CELL 6 — COMPARISON FUNCTIONS ACROSS EXPERIMENTS
# ============================================================

def plot_residual_heatmap(results, loaded_params, fig_path,
                             filename='K_residual_heatmap'):
    n = len(results)
    ncols = min(n, 3); nrows = int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 4.5*nrows),
                             sharex=True, sharey=True, constrained_layout=True)
    axes = np.array(axes).flatten()
    fmags = [np.sqrt(r['FU_pred']**2 + r['FV_pred']**2) for r in results]
    vmin_g = min(fm[fm>0].min() for fm in fmags if (fm>0).any())
    vmax_g = max(fm.max() for fm in fmags)
    norm_g = LogNorm(vmin=vmin_g, vmax=vmax_g)
    x_vals = results[0]['x'].flatten(); t_vals = results[0]['t'].flatten()
    extent = [x_vals.min(), x_vals.max(), t_vals.min(), t_vals.max() ]
    ims = []
    for i, (res, exp, fm) in enumerate(zip(results, loaded_params, fmags)):
        ax = axes[i]
        im = ax.imshow(fm, origin='lower', aspect='auto', extent=extent,
                       cmap=cmocean.cm.thermal_r, norm=norm_g)
        ims.append(im)
        ax.set_title(exp.get('desc'))
        ax.set_xlabel(r'$x$'); ax.set_ylabel(r'$t$')
    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    for ax in axes[:n]: ax.label_outer()
    for k, ax in enumerate(axes[:n]):
        row, col = divmod(k, ncols)
        has_below = (row + 1 < nrows) and ((row + 1) * ncols + col < n)

        if not has_below:
            ax.tick_params(axis='x', labelbottom=True)
            ax.set_xlabel(r"$x$")

        if col == 0:
            ax.set_ylabel(r"$t$")
    fig.colorbar(ims[-1], ax=axes[:n], fraction=0.02, pad=0.02,
                 label=r'$\| \boldsymbol{\mathcal{R}} \| = \sqrt{F_{\hat{u}}^2 + F_{\hat{v}}^2}$')
    _save_fig(fig, fig_path, filename); plt.show(); plt.close(fig)


def plot_striae_comparison(results, loaded_params, fig_path, n_levels=8,
                           filename='L_striae_comparison'):
    """
    Figure 1xN with the absolute error maps |psi_pred-psi_exact|
    on a logarithmic scale for all strategies.
    Shared global color scale for direct comparison.

    Allows visually verifying whether the low-error striae
    are independent of the boundary configuration.

    Output: {filename}.pdf / .png
    """
    n      = len(results)
    errors = [np.abs(r['H_star'] - r['H_pred']) for r in results]

    # Shared global scale
    vmin_g = min(e[e > 0].min() for e in errors if (e > 0).any())
    vmax_g = max(e.max() for e in errors)
    norm_g = LogNorm(vmin=vmin_g, vmax=vmax_g)

    x_vals = results[0]['x'].flatten()
    t_vals = results[0]['t'].flatten()
    extent = [x_vals.min(), x_vals.max(),
              t_vals.min(), t_vals.max()]

    ncols = min(n, 3); nrows = int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 4.5*nrows),
                             constrained_layout=True, sharex=True, sharey=True)
    axes = np.array(axes).flatten()
    ims = []
    for i, (res, exp, err, ax) in enumerate(
            zip(results, loaded_params, errors, axes)):
        im = ax.imshow(err, origin='lower', aspect='auto',
                       extent=extent,
                       cmap=cmocean.cm.thermal_r, norm=norm_g)
        ims.append(im)
        strat = exp.get('desc')
        ax.set_title(strat)
        ax.set_xlabel(r'$x$')
        ax.set_ylabel(r'$t$')
    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    for ax in axes[:n]: ax.label_outer()
    for k, ax in enumerate(axes[:n]):
        row, col = divmod(k, ncols)
        has_below = (row + 1 < nrows) and ((row + 1) * ncols + col < n)

        if not has_below:
            ax.tick_params(axis='x', labelbottom=True)
            ax.set_xlabel(r"$x$")

        if col == 0:
            ax.set_ylabel(r"$t$")

    fig.colorbar(ims[-1], ax=axes, fraction=0.02, pad=0.02,
                 label=r'$\left||\hat{\psi}(x,t;\boldsymbol{\theta}^{*})| - |\psi(x,t)|\right|$')
    _save_fig(fig, fig_path, filename)
    plt.show(); plt.close(fig)

def plot_time_slices_comparison(results, loaded_params, fig_path, filename='M_time_slices_comp',
                         linestyles=None, n_cuts=9, x_text_offset=0.8,
                         y_scale=1.0
                         ):
    """
    Generates a waterfall overlaying the exact solution and the predictions
    (one per experiment configuration) over n_cuts time slices, with a vertical
    offset between slices.

    Parameters
    ----------
    results : list of dict
        See the module docstring.
    loaded_params : list of dict
        Metadata per run.
    fig_path : str
        Output directory (passed to _save_fig).
    filename : str
        File base name.
    linestyles : list or None
        Line styles per configuration. If None, '--' is used for all.
    n_cuts : int
        Number of time slices.
    x_text_offset : float
        Horizontal offset for the 't = ...' label.
    y_scale : float
        Vertical separation between slices (default 2.0). Adjust if the
        breather amplitude is large (e.g. up to 2.4 for Bound).
    exact_source : int
        Index of the element of results from which Exact_h is taken.
    """
    # --- Minimal validation -------------------------------------------------
    n_exp = len(results)
    colors = sns.color_palette('colorblind', n_exp)
    if len(loaded_params) != n_exp:
        raise ValueError(
            f"len(loaded_params)={len(loaded_params)} != len(results)={n_exp}"
        )
    # if len(colors) < n_exp:
    #     raise ValueError(
    #         f"colors must have >= {n_exp} entries (received {len(colors)})"
    #     )
    if linestyles is None:
        linestyles = ['--'] * n_exp
    elif len(linestyles) != n_exp:
        raise ValueError(
            f"linestyles must have {n_exp} entries if specified"
        )
    # --- Spatial and temporal axes --------------------------------------
    x_vals = results[0]['x'].flatten()
    t_vals = results[0]['t'].flatten()
    Exact_h = results[0]['Exact_h']  # (Nx, Nt)
    cut_idx = np.linspace(0, len(t_vals) - 1, n_cuts, dtype=int)

    # --- Figure ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10,8))

    for i, idx in enumerate(cut_idx):
        offset = i * y_scale
        # Exact reference (gray, solid). Plotted once per slice.
        ax.plot(x_vals, Exact_h[:, idx] + offset,
                color='gray', lw=1.5, alpha=0.8)# zorder=10
        # Predictions of each experiment/configuration.
        for j, (res, params, col, ls) in enumerate(
                zip(results, loaded_params, colors, linestyles)):
            ax.plot(x_vals, res['H_pred'][idx, :] + offset,
                    linestyle=ls, color=col, lw=1.5,
                    alpha=0.9, zorder=5)
        # Label t = ... on the left side.
        ax.text(x_vals.min() - x_text_offset, offset,
                rf'$t={t_vals[idx]:.2f}$',
                va='center', ha='right')

    # --- Axes and aesthetics ---------------------------------------------------
    ax.set_xlim([x_vals.min(), x_vals.max()])
    ax.set_ylim([-0.1, (n_cuts + 0.5) * y_scale])
    ax.set_yticks([])
    ax.set_xlabel(r'$x$')
    # Change the title according to the experiment:
    ax.set_title(r'Time slices of $|\psi(x,t)|$ by sampling strategy')

# --- Single legend outside the canvas ------------------------------------
    # Exact element (first) + one element per BC configuration.
    legend_handles = [
        plt.Line2D([0], [0], color='gray', lw=1.5, alpha=0.8)
    ]
    legend_labels = ['Exact']
    for j, (params, col, ls) in enumerate(zip(loaded_params, colors, linestyles)):
        label = params.get('desc', f'Exp. {j+1}')
        legend_handles.append(
            plt.Line2D([0], [0], color=col, lw=1.5, linestyle=ls,
                       alpha=0.9)
        )
        legend_labels.append(label)

    fig.legend(
        legend_handles, legend_labels,
        bbox_to_anchor=(0.92, 0.5),  # outside, to the right of the panel
        loc='center left',
        borderaxespad=0.,
        fontsize=10,
        framealpha=0.95,
        title='Strategy', # Change according to the experiment or batch
        title_fontsize=11,
    )

    # --- Saving ----------------------------------------------------------
    # _save_fig is assumed to be available in the notebook scope.
    # If it is not, replace it with a manual fig.savefig(...).
    try:
        _save_fig(fig, fig_path, filename)
    except NameError:
        # Fallback if _save_fig does not exist.
        import os
        os.makedirs(fig_path, exist_ok=True)
        for ext in ('pdf', 'png'):
            fig.savefig(f'{fig_path}/{filename}.{ext}',
                        bbox_inches='tight', dpi=200)

    plt.show()
    plt.close(fig)

def plot_sampling_panel(results, loaded_params, fig_path,
                        filename='N_sampling_panel',
                        bg_field='H_star', bg_alpha=0.55):
    """
    N-panel figure with the spatial distribution of T_f for each
    sampling strategy. Common background: reference |psi(x,t)|
    (identical in all panels); overlay: IC/BC (black x) and
    collocation points (red).

    Parameters
    ----------
    results       : list of dict — one per strategy (same base case).
    loaded_params : list of dict — metadata; 'strategy' or 'desc' is used.
    fig_path      : str
    filename      : str
    bg_field      : 'H_star' (reference) or 'H_pred' (prediction).
                    Recommended: 'H_star' so that the background is the same
                    in all panels.
    bg_alpha      : background transparency (0-1). Lower -> points more visible.
    """
    n = len(results)
    H_vals = [res[bg_field] for res in results]

    vmax_g = max(h.max() for h in H_vals)
    vmin_g = min(h[h > 0].min() for h in H_vals if (h > 0).any())
    norm_g = LogNorm(vmin=vmin_g, vmax=vmax_g)

    x_vals = results[0]['x'].flatten()
    t_vals = results[0]['t'].flatten()

    extent = [x_vals.min(), x_vals.max(),
              t_vals.min(), t_vals.max()]

    ncols = min(n, 2)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5 * ncols, 4.5 * nrows),
                             constrained_layout=True,
                             sharex=True, sharey=True)
    axes = np.array(axes).flatten()

    ims = []
    for i, (res, exp, ax) in enumerate(zip(results, loaded_params, axes)):

        # Background: reference |psi| (same in all panels)
        im = ax.imshow(res[bg_field], extent=extent,
                       origin='lower', aspect='auto',
                       cmap=cmocean.cm.haline,
                       norm=norm_g,
                       alpha=bg_alpha)
        ims.append(im)

        ax.autoscale(False)

        # IC/BC as black x
        if 'X_u_train' in res:
            Xu = res['X_u_train']
            ax.plot(Xu[:, 0], Xu[:, 1], 'kx',
                    markersize=4, clip_on=False,
                    label=f'IC/BC ({Xu.shape[0]})')

        # Collocation points in red
        if 'X_f_train' in res:
            Xf = res['X_f_train']
            ax.scatter(Xf[:, 0], Xf[:, 1],
                       c='red', s=1.5, alpha=0.5,
                       zorder=2, rasterized=True,
                       label=f'Collocation ({Xf.shape[0]})')

        strat = exp.get('desc', f'exp{loaded_params.index(exp)}')
        ax.set_title(strat)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.85)

    # Hide unused axes
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    # Axis labels: only on border rows/columns
    for k, ax in enumerate(axes[:n]):
        row, col = divmod(k, ncols)
        has_below = (row + 1 < nrows) and ((row + 1) * ncols + col < n)
        if not has_below:
            ax.set_xlabel(r'$x$')
            ax.tick_params(axis='x', labelbottom=True)
        if col == 0:
            ax.set_ylabel(r'$t$')

    # Shared colorbar for the background
    fig.colorbar(ims[-1], ax=axes.tolist(),
                 fraction=0.02, pad=0.02,
                 label=r'$|\psi(x,t)|$')

    _save_fig(fig, fig_path, filename)
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# CELL 7 — MODULAR EXECUTION
# ============================================================
# Uncomment the lines of the figures to be generated.
# Each function is independent.
#
# INDIVIDUAL FIGURES (one experiment at a time, index idx):
#   plot_field_error         -> field u / v / |psi|  (1×3)
#   plot_loss_total          -> total loss Adam + L-BFGS
#   plot_loss_components     -> loss by components IC / BC / Res
#   plot_striae_isolines     -> error isolines (striae pattern)
#
# COMPARATIVE FIGURES (all experiments at once):
#   plot_loss_comparison     -> total loss compared across experiments
#   plot_l2_instantaneous    -> compared instantaneous L2 error
#   summary_table            -> CSV table with L2, times and iterations
#   plot_norm_conservation   -> exact vs predicted N(t) + ΔN_rel(t)
#   summary_spearman_table   -> CSV + JSON table of correlations
#   plot_residual_heatmap    -> comparison of residuals across experiments
#   plot_striae_comparison   -> comparison of striae patterns
#   plot_time_slices_comparison -> comparative waterfall of time slices
#   plot_sampling_panel      -> comparative panel of sampling strategies
#
# ANIMATIONS:
#   animate_solution_vs_error -> MP4 profile |psi| + absolute error in t
# ============================================================


## --- Individual figures -------------------------------------------------

# plot_norm_conservation and plot_time_slices can be defined for a single
# experiment when reading the files; it is not necessary to read a particular index.
# We load a single experiment and run the functions automatically; only the
# filename needs to be changed to save the files.
# For plot_time_slices_comparison a single line style must be defined so that
# the function does not raise warnings (the number of linestyles is the total number of
# compared experiments).

idx = min(1, n_exp - 1)   # second loaded experiment, or the only one
res = results[idx]
lbl = labels[idx]
dsc = descs[idx]

plot_field_error(res, lbl, dsc, SAVE_PATH, field='h')
plot_field_error(res, lbl, dsc, SAVE_PATH, field='u')
plot_field_error(res, lbl, dsc, SAVE_PATH, field='v')
plot_loss_total(res, lbl, dsc, SAVE_PATH)
plot_loss_components(res, lbl, dsc, SAVE_PATH)

plot_striae_isolines(res, lbl, dsc, SAVE_PATH, n_levels=8)


## --- Comparative figures ----------------------------------------------------

summary_table(results, loaded_params, SAVE_PATH, filename=f'G_summary_table_{case_tag}_Sampling')
plot_loss_comparison(results, loaded_params, SAVE_PATH, filename=f'D_loss_comp_{case_tag}_Sampling')

## Conservation of N(t) — 3 panels when show_boundary_flux=True: exact vs predicted N(t), ΔN_rel(t) in log scale, and boundary-flux residual
plot_norm_conservation(results, loaded_params, SAVE_PATH,
                       equation=equation, show_boundary_flux=True,
                       filename=f'I_norm_conservation_{case_tag}_Sampling')
plot_residual_heatmap(results, loaded_params, SAVE_PATH,
                         filename=f'K_res_heatmap_{case_tag}_Sampling')
plot_striae_comparison(results, loaded_params, SAVE_PATH,
                       filename=f'L_striae_comparison_{case_tag}_Sampling')
#
# Waterfall of 5 BC configurations:
# linestyles_5bc = ['--', '-.', ':', '--', '-.']
linestyles_samp = ['--', '-.', ':', '--'][:n_exp]
# linestyles_single = ['--'] # Para un solo experimento
plot_time_slices_comparison(results, loaded_params, SAVE_PATH,
    linestyles=linestyles_samp,
    filename=f'M_time_slices_{case_tag}_Sampling',
    n_cuts=9,
    y_scale=1.25,          # vertical spacing between slices; increase for large amplitudes (e.g. breathers)
    x_text_offset=0.1
)
plot_sampling_panel(results, loaded_params, SAVE_PATH,
                    filename=f'N_sampling_panel_{case_tag}', bg_alpha=0.8)

## --- Correlations — loop over all experiments ----------------------
#
## CSV + JSON table.
#
tabla_corr = []
for res_i, lbl_i, dsc_i in zip(results, labels, descs):
    fila = compute_spearman(res_i, lbl_i, dsc_i,
                            compute_gradient_magnitude, equation=equation)
    tabla_corr.append(fila)
summary_spearman_table(tabla_corr, SAVE_PATH,
                       filename=f'H_spearman_{case_tag}_Sampling')
